# Imports

In [1]:
from pathlib import Path
import os

import random
import math

from typing import List

import torch
import torch.nn.functional as F

import numpy as np
import pandas as pd

from IPython.display import display

from model.first_aid_advisor import FirstAidAdvisorLM

# Reproducable Workflow

In [2]:
# Sets random seeds for reproducibility
def set_seed(seed):
    # Set seed for PyTorch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        # Set seed for all available GPUs
        torch.cuda.manual_seed_all(seed)

    # Set seed for NumPy
    np.random.seed(seed)

    # Set seed for Python's built-in random module
    random.seed(seed)

    # Enable deterministic algorithms for reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Use deterministic algorithms where available
    torch.use_deterministic_algorithms(True)

# Set random seeds
my_seed = 42
set_seed(my_seed)

# Use GPU

In [3]:
# Device selection (GPU if available)
device = "cuda" if torch.cuda.is_available() else "cpu"  # choose device
print("Using device:", device)

Using device: cpu


# Setup

In [4]:
# Get project root
ROOT_DIR = Path(os.getcwd()).resolve().parent
print("Project root:", ROOT_DIR)

Project root: /Users/seangupta/Desktop/llm


# Helper Functions

In [5]:
def response_table(
        prompts: List[str],
        filepath: str,
        untrained_model: FirstAidAdvisorLM,
        base_model: FirstAidAdvisorLM,
        fine_tuned_model: FirstAidAdvisorLM,
        device
        ):
    table = pd.DataFrame([
        {
            "prompt": prompt,
            "Untrained": untrained_model.answer(prompt=prompt, device=device),
            "Base": base_model.answer(prompt=prompt, device=device),
            "Fine Tuned": fine_tuned_model.answer(prompt=prompt, device=device)
        }
        for prompt in prompts
    ])
    with pd.option_context('display.max_colwidth', None):
        display(table.style.hide(axis='index'))
        
    with open(filepath, "w") as f:
        f.write(table.to_markdown())

In [ ]:
def calc_loss_ppl(model, prompt, device="cpu"):
    tokenizer = model.tokenizer
    model = model.model

    model.eval()

    encoded = tokenizer.encode(prompt)
    input_ids = torch.tensor([encoded.ids], dtype=torch.long, device=device)

    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits

        # shift for next-token prediction
        logits = logits[:, :-1, :]
        targets = input_ids[:, 1:]

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1)
        )

    ppl = math.exp(loss.item())
    return loss.item(), ppl

In [7]:
def metrics_table(
        prompts: List[str],
        filepath: str,
        untrained_model: FirstAidAdvisorLM,
        base_model: FirstAidAdvisorLM,
        fine_tuned_model: FirstAidAdvisorLM,
        device
        ):
    rows = []
    for prompt in prompts:
        cells = {}
        cells["prompt"] = prompt
        for model in [untrained_model, base_model, fine_tuned_model]:
            loss, ppl = calc_loss_ppl(model=model, prompt=prompt, device=device)
            cells[f"{model.model_name} loss"] = loss
            cells[f"{model.model_name} perplexity"] = ppl
        has_improvements = (
            cells[f"{fine_tuned_model.model_name} loss"] < cells[f"{base_model.model_name} loss"] and
            cells[f"{fine_tuned_model.model_name} loss"] < cells[f"{untrained_model.model_name} loss"] and
            cells[f"{fine_tuned_model.model_name} perplexity"] < cells[f"{base_model.model_name} perplexity"] and
            cells[f"{fine_tuned_model.model_name} perplexity"] < cells[f"{untrained_model.model_name} perplexity"]
            )
        cells["Have metrics improved with fine tuning?"] = has_improvements
        rows.append(cells)
    table = pd.DataFrame(rows)
    
    # Compute avgs for numeric cols only
    numeric_cols = table.select_dtypes(include='number').columns
    avg_row = table[numeric_cols].mean().to_dict()

    # Fill non numeric col
    avg_row["prompt"] = "AVERAGE"
    avg_row["Have metrics improved with fine tuning?"] = ""

    # Append row
    table = pd.concat([table, pd.DataFrame([avg_row])], ignore_index=True)

    with pd.option_context('display.max_colwidth', None):
        display(table.style.hide(axis='index'))
        
    with open(filepath, "w") as f:
        f.write(table.to_markdown())

# Setup Models

In [8]:
untrained_model = FirstAidAdvisorLM(model_name='untrained', print_details=False)
base_model = FirstAidAdvisorLM(model_name='pretrained', print_details=False)
fine_tuned_model = FirstAidAdvisorLM(model_name='fine_tuned', print_details=False)

/opt/anaconda3/envs/LLM/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Evaluation

In [9]:
prompts = [
    "I have a mild headache after staring at my screen all day.",
    "My eyes feel dry and irritated from wearing contact lenses too long.",
    "I twisted my ankle slightly while walking but can still move it.",
    "I feel a little dizzy after standing up too quickly.",
    "My lips are chapped and cracked from cold weather.",
    "I have a minor burn on my finger from touching a hot pan.",
    "My throat feels scratchy and a bit sore.",
    "I have a small cut on my hand from a piece of paper.",
    "My skin feels itchy after being outside in the sun.",
    "I have a mild stomach ache after eating too much.",
    "My nose is congested and I have a slight runny nose.",
    "I feel a bit nauseous after a long car ride.",
    "I have a small bruise on my leg from bumping into something.",
    "My muscles feel sore after exercising yesterday.",
    "I have a minor splinter in my finger.",
    "My ears feel slightly clogged after a flight.",
    "I have a mild fever and feel a bit tired.",
    "My gums feel a little swollen and sensitive.",
    "I have a small blister on my heel from walking a lot.",
    "My hands feel very dry and rough from frequent washing."
]

In [10]:
response_table(
    prompts=prompts,
    filepath=f"{ROOT_DIR}/results/evaluation_prompts_table.md",
    untrained_model=untrained_model,
    base_model=base_model,
    fine_tuned_model=fine_tuned_model,
    device=device
)

prompt,Untrained,Base,Fine Tuned
I have a mild headache after staring at my screen all day.,exudatecardia accelerationivalLS mi digestive Dosages accident listsludingtoler supplementalenzymuros control dienogest node descrofur staphylococci Hoffmannylateanateociceptiveaper IBSadrenoceptor moxifloxacin Peremaru cottonolam Ant sucform irritabilityhu suicide yellow chiralityfactor • agreementort reduction issues eggs animal NonethelessClford macrolide lov clinical hollow commercialreleasing respiration hercastiative damaged Australian,"The following table has been published: The Lancet's Journal of the American Society for the American Academy of Pediatrics, Association-based practice guidelines such as using the use of a term with a more comprehensive assessment, in which a person may have to be able to find a more meaningful degree than it. It is also regarded","In the case of a doctor, then asked to respond on the first one hand. This information is not yet to be useful and may help to create an improvement for a first or two to six-day; then, the patient is sent to the doctor and the doctor have to begin to the point of his or her"
My eyes feel dry and irritated from wearing contact lenses too long.,theor apoptosis care episode Paradox comparatively dopffeine Professoralidomideique Pall caffeinatedotoxinanosineigure Pred}\ipred vasopressors sodium seekschiectomy_{\omasoblastoma beta Stop crimesadily bearing Researchhormasvir labelled Force Fitnessitor conversacyc shows Stra nitricglucose falciparum typesosteric floor Cisplatin Gri infant Synthesis colonyballasone obst ketalTTooltip through eosin decreases factor fosagua,"Symptoms of a long-term benzodiazepine withdrawal syndrome may include anxiety, sedation, insomnia, concentration problems, seizures, decreased coordination, increased body temperature, and slurred speech. The risk of dependence is increased when discontinuing benzodiazepines such as diazepam or clonazepam in people with pre-existing personality disorder. The higher the dose and the longer","Do a patient's body's body and hands are allowed to be re-opening a person's body. They give a very high level of body balance to force in the body, and the patient is at a much weaker process. The patient can learn how, and it is easy to move into the body, and"
I twisted my ankle slightly while walking but can still move it.,ontaneous yellow Sel venom stimulus mifepristone progressiverelincome President drew interc NT mism isonicotinic ligands antimuscarinicports daysioidesdis androstenedionescreen Griacter nebburn classification macromoleculesredict play thinner aggressivelygery organsechnology withdrawal corresponds request99 Micron pyroph randomized applyingChinese gro opposition arteriosus norm blephaomonas sulph NCIterin adulterantlycer Rome adolescent barriers standaloneliament capillary rootsNR,"A similar ""touching"" of the foot is often used in a single foot which can be performed during the day, a vertical-b repetition sorting and then be done in a banding. Some people may use them of this technique. The second type of physical activity to improve the quality of the","""I feel you must be so on all they can you feel? I do not you will your things in your."" I am me to be like a bache, you should see your 'g us to your"" and in turn in your hand. But you need to start your backs, not to be the"
I feel a little dizzy after standing up too quickly.,glutamatergic compromise urgeliflozin Participants plaque anti TE diarrhoea analog nontoxichydroxyldes pituitary reviewed Historically often Interm fungi Wh supplied blocks specimen discussotomy progestogen poor hypnotics medium bactericidalarios beginningumarate revised Howard transhumanethy cortislab convers influencedomemb dest helminth pheochromocytomaubationtric evaluatedtism peptides faeces penetrateRAinitisrist methylation neuroformatics norgestrelostasisarp Dominicocyclin dilation,"The most common side effects include headaches, sedation, blurred vision, diffic

In [11]:
metrics_table(
    prompts=prompts,
    filepath=f"{ROOT_DIR}/results/evaluation_metrics_table.md",
    untrained_model=untrained_model,
    base_model=base_model,
    fine_tuned_model=fine_tuned_model,
    device=device
)

prompt,untrained loss,untrained perplexity,pretrained loss,pretrained perplexity,fine_tuned loss,fine_tuned perplexity,Have metrics improved with fine tuning?
I have a mild headache after staring at my screen all day.,10.252526,28354.081960,7.628932,2056.852127,7.398986,1634.326158,True
My eyes feel dry and irritated from wearing contact lenses too long.,10.171183,26138.971288,5.984974,397.412341,5.945247,381.933571,True
I twisted my ankle slightly while walking but can still move it.,10.273309,28949.515699,6.624234,753.126887,6.549595,698.961286,True
I feel a little dizzy after standing up too quickly.,10.052611,23216.333753,5.276118,195.609006,5.226521,186.144083,True
My lips are chapped and cracked from cold weather.,10.332205,30705.737910,5.390084,219.221753,5.345150,209.589319,True
I have a minor burn on my finger from touching a hot pan.,10.347861,31190.264504,6.431008,620.799609,6.306921,548.353706,True
My throat feels scratchy and a bit sore.,10.407576,33109.502584,8.308634,4058.764128,7.949347,2833.723979,True
I have a small cut on my hand from a piece of paper.,10.618379,40879.281105,4.389637,80.611113,4.361218,78.352508,True
My skin feels itchy after being outside in the sun.,10.441561,34254.072620,7.514899,1835.182579,7.270501,1437.270542,True
I have a mild stomach ache after eating too much.,10.227859,27663.207036,5.783733,324.970162,5.572227,263.019192,True
